# Microsoft Agent Framework — Azure OpenAI (Responses API)

या कोड नमुन्यात, तुम्ही **Microsoft Agent Framework (MAF)** वापरून **Azure OpenAI** द्वारे समर्थित साधा एजंट तयार कराल जो **Responses API** वापरतो.

> **मायग्रेशन नोट:** हा नमुना पूर्वी Semantic Kernel सह GitHub Models वापरत होता. याला Microsoft Agent Framework मध्ये स्थलांतरित केले गेले आहे, आणि GitHub Models (ज्याचे मासिक जुलै २०२६ मध्ये निवृत्ती होणार आहे) याच्या जागी Azure OpenAI वापरण्यात आले आहे, जे Responses API ला समर्थन देते. MAF मधील `OpenAIChatClient` Azure OpenAI च्या स्थिर `/openai/v1/` एंडपॉइंटसाठी उद्दिष्ट ठरला आहे आणि डिफॉल्टने Responses API वापरतो.

या नमुन्याचा हेतू पुढील कोड नमुन्यांमध्ये विविध एजंटिक पॅटर्न्स अमलात आणताना वापरल्या जाणाऱ्या पायऱ्या दाखवणे आहे.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## आवश्यक पायथन पॅकेजेस आयात करा


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## उपकरणाची व्याख्या करणे

Microsoft Agent Framework मध्ये, **उपकरण** म्हणजे एक सामान्य Python फंक्शन जे `@tool` ने सजवलेले असते आणि ज्याला एजंट कॉल करू शकतो. खाली आपण एक उपकरण परिभाषित केले आहे जे एक यादृच्छिक सुट्टीच्या ठिकाणाचे स्थान परत करते आणि मागील ठिकाणाला पुन्हा वापरण्यापासून बचाव करते.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## एजंट तयार करताना

येथे, आपण `TravelAgent` नावाचा एजंट तयार करतो.

या उदाहरणात, आपण अगदी मूलभूत सूचना वापरतो. एजंटच्या वर्तनात कशी बदल होतो हे पाहण्यासाठी या सूचनांमध्ये मोकळेपणाने बदल करा.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## एजंट चालू करणे

आता आपण एजंट चालवू शकतो. आपण एक `AgentSession` तयार करतो जेणेकरून एजंट संवादाच्या वळणांमध्ये संभाषण लक्षात ठेवू शकेल, नंतर दोन `user_inputs` पाठवतो. पहिला प्रवासासाठी विचारतो; दुसरा म्हणतो की वापरकर्त्याला सल्ला आवडला नाही आणि दुसरा विचारतो — एजंट सत्र इतिहास आणि `get_random_destination` साधन वापरून प्रतिसाद देतो.

आपण एजंट वेगवेगळ्या प्रतिसादांमध्ये कसा प्रतिक्रिया देतो ते पाहण्यासाठी या संदेशांमध्ये बदल करू शकता. प्रतिसाद **टोकन-दर-टोकन** प्रवाहित केले जातात.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
